# Auditoria do Stroke Prediction Dataset

Esta etapa verifica o CSV fornecido antes de qualquer divisão, imputação ou treinamento. A referência metodológica é Dritsas e Trigka (2022), [DOI 10.3390/s22134670](https://doi.org/10.3390/s22134670). O artigo relata 3.254 participantes com idade > 18 anos; aqui confrontamos essa descrição com o arquivo disponível.

In [1]:
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from IPython.display import display

# O caminho funciona ao abrir o notebook pela raiz ou pela pasta notebooks/.
candidates = [Path.cwd() / 'healthcare-dataset-stroke-data.csv',
              Path.cwd().parent / 'healthcare-dataset-stroke-data.csv']
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError('CSV não encontrado na pasta atual ou na pasta acima.')

data = pd.read_csv(data_path)
print('Arquivo:', data_path.resolve())
print('SHA-256:', sha256(data_path.read_bytes()).hexdigest())
print('Python/pandas:', __import__('sys').version.split()[0], version('pandas'))
print('Dimensões:', data.shape)
display(data.head(3))

Arquivo: D:\Usuario\Documentos\GitHub\XAI - AVC\healthcare-dataset-stroke-data.csv
SHA-256: 644d473b05d2797006bd94865e4f8bb057f0c721617911613c82c8fcfc707420
Python/pandas: 3.14.2 3.1.0
Dimensões: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1


## Estrutura, ausências e integridade

`pandas.read_csv` interpreta a string `N/A` de `bmi` como valor ausente. Por isso, a inspeção usa `isna()`, além da contagem de categorias. `id` identifica linhas e não será atributo do modelo.

In [2]:
expected_columns = ['id', 'gender', 'age', 'hypertension', 'heart_disease',
                    'ever_married', 'work_type', 'Residence_type',
                    'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']
assert data.columns.tolist() == expected_columns, 'Estrutura do CSV mudou.'
assert set(data['stroke'].dropna().unique()) == {0, 1}, 'Alvo inesperado.'

print('Tipos:')
display(data.dtypes.rename('dtype').to_frame())
print('Valores ausentes no CSV completo:')
display(data.isna().sum().rename('ausentes').to_frame())
print('IDs duplicados:', data['id'].duplicated().sum())
print('Linhas duplicadas (incluindo ID):', data.duplicated().sum())
print('Classe no CSV completo:')
display(data['stroke'].value_counts().sort_index().rename('registros').to_frame())

Tipos:


,dtype
id,int64
gender,str
age,float64
hypertension,int64
heart_disease,int64
ever_married,str
work_type,str
Residence_type,str
avg_glucose_level,float64
bmi,float64


Valores ausentes no CSV completo:


,ausentes
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,201


IDs duplicados: 0
Linhas duplicadas (incluindo ID): 0
Classe no CSV completo:


,registros
stroke,
0,4861
1,249


## Coorte com idade > 18 anos

O filtro abaixo é o único critério de seleção explícito no artigo. As categorias `Unknown` e `Other` são mantidas na auditoria para evitar exclusões implícitas.

In [3]:
adults = data.loc[data['age'] > 18].copy()
print('Adultos:', len(adults))
print('Classe positiva:', int(adults['stroke'].sum()))
print('Prevalência positiva:', f"{adults['stroke'].mean():.2%}")
print('Distribuição do alvo:')
display(adults['stroke'].value_counts().sort_index().rename('registros').to_frame())
print('Valores ausentes entre adultos:')
display(adults.isna().sum().rename('ausentes').to_frame())
for column in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
    print(f'Categorias de {column}:')
    display(adults[column].value_counts(dropna=False).rename('registros').to_frame())
print('Faixas numéricas entre adultos:')
display(adults[['age', 'avg_glucose_level', 'bmi']].describe().T)

Adultos: 4194
Classe positiva: 247
Prevalência positiva: 5.89%
Distribuição do alvo:


,registros
stroke,
0,3947
1,247


Valores ausentes entre adultos:


,ausentes
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,180


Categorias de gender:


,registros
gender,
Female,2543
Male,1650
Other,1


... (6 outputs left out)

Categorias de smoking_status:


,registros
smoking_status,
never smoked,1727
formerly smoked,857
Unknown,835
smokes,775


Faixas numéricas entre adultos:


,count,mean,std,min,25%,50%,75%,max
age,4194.0,50.662852,17.532031,19.00,37.00,51.00,64.000,82.00
avg_glucose_level,4194.0,108.632995,48.004093,55.12,77.42,92.36,116.225,271.74
bmi,4014.0,30.493921,7.222288,11.30,25.50,29.30,34.200,92.00


## Reconciliação com o artigo

Os filtros exploratórios abaixo **não** definem a coorte do experimento. Eles testam se duas exclusões plausíveis explicariam a amostra publicada; nenhuma deve ser aplicada só para alcançar 3.254.

In [4]:
known_smoking = adults.loc[adults['smoking_status'] != 'Unknown']
known_smoking_complete_bmi = known_smoking.loc[known_smoking['bmi'].notna()]
comparison = pd.DataFrame([
    {'recorte': 'Artigo (declarado)', 'registros': 3254, 'stroke_1': None},
    {'recorte': 'CSV: idade > 18', 'registros': len(adults), 'stroke_1': int(adults['stroke'].sum())},
    {'recorte': 'Idade > 18 e tabagismo conhecido', 'registros': len(known_smoking), 'stroke_1': int(known_smoking['stroke'].sum())},
    {'recorte': 'Também com BMI observado', 'registros': len(known_smoking_complete_bmi), 'stroke_1': int(known_smoking_complete_bmi['stroke'].sum())},
])
display(comparison)
assert (len(data), len(adults), int(adults['stroke'].sum())) == (5110, 4194, 247)
assert (int(data['bmi'].isna().sum()), int(adults['bmi'].isna().sum())) == (201, 180)
assert (len(known_smoking), len(known_smoking_complete_bmi)) == (3359, 3224)
print('Conferência das contagens da cópia fornecida: OK')
print('Diferença para a amostra publicada:', len(adults) - 3254, 'registros')

,recorte,registros,stroke_1
0,Artigo (declarado),3254,NaN
1,CSV: idade > 18,4194,247.0
2,Idade > 18 e tabagismo conhecido,3359,202.0
3,Também com BMI observado,3224,180.0


Conferência das contagens da cópia fornecida: OK
Diferença para a amostra publicada: 940 registros
